In [7]:
import requests
from bs4 import BeautifulSoup

# ニュース一覧ページのURL
url = "https://www.yahoo.co.jp/"

# ページのHTMLを取得
headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers, verify=False)
response.encoding = response.apparent_encoding  # 文字コードを自動判別して設定

# HTMLをBeautifulSoupでパース
soup = BeautifulSoup(response.text, "html.parser")

# ニュース一覧の情報を抽出
news_list = soup.find_all("li", class_="news-block")

# ニュース情報の取得
for news in news_list:
    # ニュースの日付を取得
    date = news.find("p", class_="date").get_text(strip=True) if news.find("p", class_="date") else "N/A"
    
    # ニュースのタイトルを取得
    title = news.find("a", class_="title").get_text(strip=True)
    
    # ニュースのリンクを取得
    link = news.find("a", class_="title")["href"]

    print(f"Date: {date}")
    print(f"Title: {title}")
    print(f"Link: {link}")
    print("-" * 40)

/Users/tamoriwa/llmdev/.venv/lib/python3.13/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.yahoo.co.jp'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [4]:
# 必要なモジュールをインポート
import os
from dotenv import load_dotenv
from openai import OpenAI

# 環境変数の取得
load_dotenv("../.env")

# OpenAI APIクライアントを生成
client = OpenAI(api_key=os.environ['API_KEY'])

# モデル名
MODEL_NAME = "gpt-4o-mini"

In [8]:
# ニュース一覧ページのURL
url = "https://www.yahoo.co.jp/"

# ページのHTMLを取得
response = requests.get(url)
response.encoding = response.apparent_encoding  # 文字コードを自動判別して設定

# HTMLをBeautifulSoupでパースし、body部分を取り出す
soup = BeautifulSoup(response.text, "html.parser")
body_html = str(soup.body)  # body部分のHTMLを文字列として取得
print(body_html) # 結果を表示して確認

<body><noscript><iframe height="0" src="https://www.googletagmanager.com/ns.html?id=GTM-TQJW5V3W" style="display:none;visibility:hidden" title="Google Tag Manager" width="0"></iframe></noscript><script>bucket_id_for_ad = ''; bucket_ids = '';</script><script>window.YAHOO = window.YAHOO || {};
              window.YAHOO.JP = window.YAHOO.JP || {};
              window.YAHOO.JP.Fp = window.YAHOO.JP.Fp || {};
              window.YAHOO.JP.Fp.ads = window.YAHOO.JP.Fp.ads || {};
              window.YAHOO.JP.Fp.ads.sidehide = true;</script><div class="_1DyDVN-3FsqGF_QKLg1M9G" data-rma-pos="GYJ" id="wrapper"><div id="ContentWrapper"><header class="_2ZLZK6xsD6ls7W_ryou_ED"><div class="_1mo4G7fxHMqUog3vJGxY2_ ult__mods" id="Masthead"><h1 class="_3YIqBohnzWyU3NQ8zb-mQI _1dr5aVDbNPF63JCS2bJhij"><a class="yMWCYupQNdgppL-NV6sMi _3sAlKGsIBCxTUbNi86oSjt" data-cl-params="_cl_vmodule:header;_cl_link:logo;_cl_position:0" href="https://www.yahoo.co.jp">Yahoo! JAPAN</a></h1><div class="_15h66qPnVy4iQzPepg

In [9]:
# LLMにニュース一覧を抽出させるプロンプトを作成
prompt = f"""
以下のHTMLから最新のニュースを抽出し、「日付、タイトル、リンク」の形式で一覧を出力してください。一覧以外は出力しないでください。

# 出力様式：
Date: 日付
Title: タイトル
Link: リンク
--------------------

#HTML:
{body_html[:5000]}
"""

# APIへリクエスト
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "user", "content": prompt},
    ],
    max_tokens=500,
    temperature=0.3
)

# LLMからの回答を表示
print(response.choices[0].message.content.strip())

```
Date: 2023年10月20日
Title: 最新のテクノロジー動向
Link: https://example.com/news1
--------------------
Date: 2023年10月19日
Title: 経済指標の発表
Link: https://example.com/news2
--------------------
Date: 2023年10月18日
Title: 環境問題に関する国際会議
Link: https://example.com/news3
--------------------
```
